# ============================================
#  Notebook 06 — Metadata & Data Dictionary
#  Memorial Sloan Kettering | Goel Lab
# ============================================

# Notebook 6: Metadata, Data Dictionary & Variable Names

**Purpose:**
1. Generate a **data dictionary** Excel file with standard headers
2. Generate a **variable names** Excel file (concise lookup)
3. Produce a **metadata summary** (dataset-level attributes)

**Data Dictionary Headers:**
| Header | Description |
|--------|-------------|
| Variable Name | Column name as it appears in the dataset |
| Label | Human-readable label |
| Description | Full description of what the variable captures |
| Data Type | Python dtype / conceptual type |
| Valid Values | Permitted coded values or range |
| Source Column | The source/origin column this derives from |
| Domain | Radiology, Pathology, Case-Level, or ID |
| Role | Source, Human Annotator, AI Annotator, Covariate, or ID |
| Missing Code | How missing data is represented |
| Notes | Additional context |

In [1]:
import os
import re
from pathlib import Path
from datetime import datetime
from collections import OrderedDict
from dotenv import load_dotenv

import pandas as pd
import numpy as np

load_dotenv()

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT     = Path(os.getenv("PROJECT_ROOT",
    r"C:\Users\jamesr4\OneDrive - Memorial Sloan Kettering Cancer Center\Documents\GitHub\llm_summarization_br_ca"))
DATA_PRIVATE_DIR = Path(os.getenv("DATA_PRIVATE_DIR", r"C:\Users\jamesr4\loc\data_private"))

# Use deidentified Excel if available, otherwise fall back to raw
DEID_PATH = DATA_PRIVATE_DIR / "deidentified" / "validation_datasheet_deidentified.xlsx"
RAW_PATH  = DATA_PRIVATE_DIR / "raw" / "merged_llm_summary_validation_datasheet.xlsx"
DATA_PATH = DEID_PATH if DEID_PATH.exists() else RAW_PATH

OUTPUT_DIR = PROJECT_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

data = pd.read_excel(DATA_PATH)
print(f"Using: {DATA_PATH.name}")
print(f"Dataset: {data.shape[0]} observations x {data.shape[1]} columns")

Using: merged_llm_summary_validation_datasheet.xlsx
Dataset: 200 observations x 49 columns


---
## 1. Element & Domain Definitions

In [2]:
ELEMENTS = OrderedDict([
    ("Lesion Size", {"source": "lesion_size_status_source", "human": "lesion_size_status_human", "ai": "lesion_size_status_ai"}),
    ("Lesion Laterality", {"source": "laterality_status_source", "human": "laterality_status_human", "ai": "laterality_status_ai"}),
    ("Lesion Location", {"source": "lesion_location_status_source", "human": "lesion_location_status_human", "ai": "lesion_location_status_ai"}),
    ("Calcifications / Asymmetry", {"source": "calcifications_asymmetry_status_source", "human": "calcifications_asymmetry_status_human", "ai": "calcifications_asymmetry_status_ai"}),
    ("Additional Enhancement (MRI)", {"source": "additional_enhancement_mri_status_source", "human": "additional_enhancement_mri_status_human", "ai": "additional_enhancement_mri_status_ai"}),
    ("Extent", {"source": "extent_status_source", "human": "extent_status_human", "ai": "extent_status_ai"}),
    ("Accurate Clip Placement", {"source": "accurate_clip_placement_status_source", "human": "accurate_clip_placement_status_human", "ai": "accurate_clip_placement_status_ai"}),
    ("Workup Recommendation", {"source": "workup_recommendation_status_source", "human": "workup_recommendation_status_human", "ai": "workup_recommendation_status_ai"}),
    ("Lymph Node", {"source": "Lymph node_status_source", "human": "Lymph node_status_human", "ai": "Lymph node_status_ai"}),
    ("Chronology Preserved", {"source": "chronology_preserved_status_source", "human": "chronology_preserved_status_human", "ai": "chronology_preserved_status_ai"}),
    ("Biopsy Method", {"source": "biopsy_method_status_source", "human": "biopsy_method_status_human", "ai": "biopsy_method_status_ai"}),
    ("Invasive Component Size (Pathology)", {"source": "invasive_component_size_pathology_status_source", "human": "invasive_component_size_pathology_status_human", "ai": "invasive_component_size_pathology_status_ai"}),
    ("Histologic Diagnosis", {"source": "histologic_diagnosis_status_source", "human": "histologic_diagnosis_status_human", "ai": "histologic_diagnosis_status_ai"}),
    ("Receptor Status", {"source": "receptor_status_source", "human": "receptor_status_human", "ai": "receptor_status_ai"}),
])

DOMAIN_LOOKUP = {}
for elem in ["Lesion Size", "Lesion Laterality", "Lesion Location",
             "Calcifications / Asymmetry", "Additional Enhancement (MRI)",
             "Extent", "Accurate Clip Placement", "Workup Recommendation",
             "Lymph Node", "Chronology Preserved"]:
    for role_col in ELEMENTS[elem].values():
        DOMAIN_LOOKUP[role_col] = "Radiology"

for elem in ["Biopsy Method", "Invasive Component Size (Pathology)",
             "Histologic Diagnosis", "Receptor Status"]:
    for role_col in ELEMENTS[elem].values():
        DOMAIN_LOOKUP[role_col] = "Pathology"

# Biopsy Method is in both domains
for role_col in ELEMENTS["Biopsy Method"].values():
    DOMAIN_LOOKUP[role_col] = "Radiology / Pathology"

print(f"Domain lookup built: {len(DOMAIN_LOOKUP)} columns mapped")

Domain lookup built: 42 columns mapped


---
## 2. Build Data Dictionary

In [3]:
def infer_role(col_name: str) -> str:
    cl = col_name.lower()
    if cl.endswith("_source"):
        return "Source (Ground Truth)"
    elif cl.endswith("_human"):
        return "Human Annotator"
    elif cl.endswith("_ai"):
        return "AI Annotator"
    elif cl in ["surgeon", "surgeon_id"]:
        return "ID"
    elif cl in ["tumor_invasive_dcis", "complex_case_status"]:
        return "Covariate"
    return "Other"

def infer_label(col_name: str) -> str:
    # Find element name
    for elem, cols in ELEMENTS.items():
        for role, cname in cols.items():
            if cname == col_name:
                role_label = {"source": "Source", "human": "Human", "ai": "AI"}[role]
                return f"{elem} ({role_label})"
    # Non-element columns
    labels = {
        "tumor_invasive_dcis": "Tumor Type (Invasive vs DCIS)",
        "complex_case_status": "Complex Case Flag",
        "surgeon": "Surgeon Identifier",
        "surgeon_id": "Surgeon Identifier",
        "mrn": "Medical Record Number",
        "patient_initials": "Patient Initials",
        "comments": "Reviewer Comments",
    }
    return labels.get(col_name, col_name.replace("_", " ").title())

def infer_description(col_name: str, role: str) -> str:
    for elem, cols in ELEMENTS.items():
        for r, cname in cols.items():
            if cname == col_name:
                if r == "source":
                    return f"Ground truth: whether '{elem}' is present (1) or absent (0) in the source documents."
                elif r == "human":
                    return f"Human annotator extraction status for '{elem}': 1=correctly extracted, 2=omitted, 3=fabricated, N/A=not applicable."
                elif r == "ai":
                    return f"AI (LLM) extraction status for '{elem}': 1=correctly extracted, 2=omitted, 3=fabricated, N/A=not applicable."
    descs = {
        "tumor_invasive_dcis": "Tumor histology classification: 1=Invasive carcinoma, 2=DCIS (ductal carcinoma in situ).",
        "complex_case_status": "Flag indicating whether the case is complex (1) or straightforward (0).",
        "surgeon": "Surgeon identifier. Links observations to the treating surgeon.",
        "surgeon_id": "De-identified surgeon identifier. Links observations to the treating surgeon.",
        "mrn": "Medical record number (PHI — redacted in deidentified version).",
        "patient_initials": "Patient initials (PHI — redacted in deidentified version).",
        "comments": "Free-text reviewer comments.",
    }
    return descs.get(col_name, "")

def infer_valid_values(col_name: str, series: pd.Series) -> str:
    for elem, cols in ELEMENTS.items():
        for r, cname in cols.items():
            if cname == col_name:
                if r == "source":
                    return "0 = Absent, 1 = Present"
                else:
                    return "1 = Correct extraction, 2 = Omission, 3 = Fabrication, N/A = Not applicable"
    if col_name == "tumor_invasive_dcis":
        return "1 = Invasive, 2 = DCIS"
    elif col_name == "complex_case_status":
        return "0 = Not complex, 1 = Complex"
    elif col_name in ["surgeon", "surgeon_id"]:
        return f"String ID; {series.nunique()} unique values"
    return str(sorted(series.dropna().unique().tolist())[:20])

def infer_missing_code(col_name: str, series: pd.Series) -> str:
    n_miss = int(series.isna().sum())
    has_na_str = False
    if series.dtype == object:
        has_na_str = series.isin(["na", "N/A", "NA"]).any()
    if n_miss == 0 and not has_na_str:
        return "None (no missing)"
    parts = []
    if n_miss > 0:
        parts.append(f"NaN ({n_miss} obs)")
    if has_na_str:
        parts.append("'na'/'N/A' = feature not applicable")
    return "; ".join(parts)

def infer_data_type(col_name: str, series: pd.Series) -> str:
    dtype = str(series.dtype)
    for elem, cols in ELEMENTS.items():
        for r, cname in cols.items():
            if cname == col_name:
                if r == "source":
                    return "Binary integer (0/1)"
                else:
                    return "Categorical (1/2/3/N/A)"
    type_map = {
        "tumor_invasive_dcis": "Binary integer (1/2)",
        "complex_case_status": "Binary integer (0/1)",
        "surgeon": "String (categorical)",
        "surgeon_id": "String (categorical)",
        "mrn": "String / Numeric (identifier)",
        "patient_initials": "String (identifier)",
        "comments": "String (free text)",
    }
    return type_map.get(col_name, dtype)

def infer_source_column(col_name: str) -> str:
    for elem, cols in ELEMENTS.items():
        for r, cname in cols.items():
            if cname == col_name and r in ["human", "ai"]:
                return cols["source"]
    return ""

def infer_notes(col_name: str) -> str:
    for elem, cols in ELEMENTS.items():
        for r, cname in cols.items():
            if cname == col_name:
                if r == "source":
                    return "Ground truth derived from source documents"
                elif r == "human":
                    return "Extracted by human annotator; compared against source"
                elif r == "ai":
                    return "Extracted by LLM (base model); compared against source"
    notes = {
        "tumor_invasive_dcis": "Case-level covariate; used for stratification",
        "complex_case_status": "Case-level covariate; may affect extraction difficulty",
        "surgeon": "Used for clustering/random effects",
        "surgeon_id": "De-identified; used for clustering/random effects",
        "mrn": "PHI — excluded from deidentified dataset",
        "patient_initials": "PHI — excluded from deidentified dataset",
        "comments": "May contain PHI in raw version",
    }
    return notes.get(col_name, "")

print("Data dictionary builder functions defined.")

Data dictionary builder functions defined.


In [4]:
# Build data dictionary
dict_rows = []
for col in data.columns:
    series = data[col]
    dict_rows.append({
        "Variable Name": col,
        "Label": infer_label(col),
        "Description": infer_description(col, infer_role(col)),
        "Data Type": infer_data_type(col, series),
        "Valid Values": infer_valid_values(col, series),
        "Source Column": infer_source_column(col),
        "Domain": DOMAIN_LOOKUP.get(col, "Case-Level" if col in ["tumor_invasive_dcis", "complex_case_status"] else ("ID" if col == "surgeon_id" else "")),
        "Role": infer_role(col),
        "Missing Code": infer_missing_code(col, series),
        "Notes": infer_notes(col),
    })

df_dict = pd.DataFrame(dict_rows)
print(f"Data dictionary: {df_dict.shape[0]} variables")
df_dict

Data dictionary: 49 variables


,Variable Name,Label,Description,Data Type,Valid Values,Source Column,Domain,Role,Missing Code,Notes
0,Unnamed: 0,Unnamed: 0,,int64,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",,,Other,None (no missing),
1,surgeon,Surgeon Identifier,Surgeon identifier. Links observations to the ...,String (categorical),String ID; 20 unique values,,,ID,None (no missing),Used for clustering/random effects
2,mrn,Medical Record Number,Medical record number (PHI — redacted in deide...,String / Numeric (identifier),"[339254, 499507, 641257, 671768, 970508, 35121...",,,Other,None (no missing),PHI — excluded from deidentified dataset
3,patient_initials,Patient Initials,Patient initials (PHI — redacted in deidentifi...,String (identifier),"['AA', 'AB', 'AD', 'AE', 'AH', 'AK', 'AM', 'AN...",,,Other,None (no missing),PHI — excluded from deidentified dataset
4,tumor_invasive_dcis,Tumor Type (Invasive vs DCIS),Tumor histology classification: 1=Invasive car...,Binary integer (1/2),"1 = Invasive, 2 = DCIS",,Case-Level,Covariate,None (no missing),Case-level covariate; used for stratification
5,complex_case_status,Complex Case Flag,Flag indicating whether the case is complex (1...,Binary integer (0/1),"0 = Not complex, 1 = Complex",,Case-Level,Covariate,None (no missing),Case-level covariate; may affect extraction di...
6,lesion_size_status_source,Lesion Size (Source),Ground truth: whether 'Lesion Size' is present...,Binary integer (0/1),"0 = Absent, 1 = Present",,Radiology,Source (Ground Truth),None (no missing),Ground truth derived from source documents
7,lesion_size_status_human,Lesion Size (Human),Human annotator extraction status for 'Lesion ...,Categorical (1/2/3/N/A),"1 = Correct extraction, 2 = Omission, 3 = Fabr...",lesion_size_status_source,Radiology,Human Annotator,NaN (2 obs); 'na'/'N/A' = feature not applicable,Extracted by human annotator; compared against...
8,lesion_size_status_ai,Lesion Size (AI),AI (LLM) extraction status for 'Lesion Size': ...,Categorical (1/2/3/N/A),"1 = Correct extraction, 2 = Omission, 3 = Fabr...",lesion_size_status_source,Radiology,AI Annotator,NaN (2 obs); 'na'/'N/A' = feature not applicable,Extracted by LLM (base model); compared agains...
9,laterality_status_source,Lesion Laterality (Source),Ground truth: whether 'Lesion Laterality' is p...,Binary integer (0/1),"0 = Absent, 1 = Present",,Radiology,Source (Ground Truth),None (no missing),Ground truth derived from source documents


---
## 3. Build Variable Names File

In [5]:
# Concise variable lookup
var_rows = []
for i, col in enumerate(data.columns):
    series = data[col]
    var_rows.append({
        "Position": i + 1,
        "Variable Name": col,
        "Label": infer_label(col),
        "Data Type": infer_data_type(col, series),
        "Domain": DOMAIN_LOOKUP.get(col, "Case-Level" if col in ["tumor_invasive_dcis", "complex_case_status"] else ("ID" if col == "surgeon_id" else "")),
        "Role": infer_role(col),
        "N Unique": int(series.nunique()),
        "N Missing": int(series.isna().sum()),
        "Pct Missing": round(series.isna().mean() * 100, 2),
    })

df_vars = pd.DataFrame(var_rows)
print(f"Variable names file: {df_vars.shape[0]} variables")
df_vars

Variable names file: 49 variables


,Position,Variable Name,Label,Data Type,Domain,Role,N Unique,N Missing,Pct Missing
0,1,Unnamed: 0,Unnamed: 0,int64,,Other,200,0,0.0
1,2,surgeon,Surgeon Identifier,String (categorical),,ID,20,0,0.0
2,3,mrn,Medical Record Number,String / Numeric (identifier),,Other,198,0,0.0
3,4,patient_initials,Patient Initials,String (identifier),,Other,149,0,0.0
4,5,tumor_invasive_dcis,Tumor Type (Invasive vs DCIS),Binary integer (1/2),Case-Level,Covariate,2,0,0.0
5,6,complex_case_status,Complex Case Flag,Binary integer (0/1),Case-Level,Covariate,2,0,0.0
6,7,lesion_size_status_source,Lesion Size (Source),Binary integer (0/1),Radiology,Source (Ground Truth),2,0,0.0
7,8,lesion_size_status_human,Lesion Size (Human),Categorical (1/2/3/N/A),Radiology,Human Annotator,4,2,1.0
8,9,lesion_size_status_ai,Lesion Size (AI),Categorical (1/2/3/N/A),Radiology,AI Annotator,4,2,1.0
9,10,laterality_status_source,Lesion Laterality (Source),Binary integer (0/1),Radiology,Source (Ground Truth),1,0,0.0


---
## 4. Build Metadata Summary

In [6]:
# Dataset-level metadata
surgeon_col = "surgeon" if "surgeon" in data.columns else "surgeon_id"
metadata = OrderedDict([
    ("Dataset Name", "Merged LLM Summary Validation Datasheet (Deidentified)"),
    ("Source File", str(DATA_PATH.name)),
    ("Date Generated", datetime.now().strftime("%Y-%m-%d %H:%M:%S")),
    ("N Observations", data.shape[0]),
    ("N Variables", data.shape[1]),
    ("N Elements (Features)", len(ELEMENTS)),
    ("N Radiology Elements", sum(1 for v in DOMAIN_LOOKUP.values() if "Radiology" in v)),
    ("N Pathology Elements", sum(1 for v in DOMAIN_LOOKUP.values() if "Pathology" in v)),
    ("Annotator Roles", "Source (Ground Truth), Human Annotator, AI (LLM) Annotator"),
    ("Source Coding", "0 = Feature absent in source document, 1 = Feature present"),
    ("Annotator Coding", "1 = Correct extraction, 2 = Omission, 3 = Fabrication, N/A = Not applicable"),
    ("Covariates", "tumor_invasive_dcis, complex_case_status"),
    ("ID Variables", surgeon_col),
    ("Total Missing Cells", int(data.isna().sum().sum())),
    ("Overall Missingness (%)", round(data.isna().sum().sum() / (data.shape[0] * data.shape[1]) * 100, 2)),
    ("Tumor Invasive Count", int((data["tumor_invasive_dcis"] == 1).sum()) if "tumor_invasive_dcis" in data.columns else "N/A"),
    ("Tumor DCIS Count", int((data["tumor_invasive_dcis"] == 2).sum()) if "tumor_invasive_dcis" in data.columns else "N/A"),
    ("Complex Case Count", int((data["complex_case_status"] == 1).sum()) if "complex_case_status" in data.columns else "N/A"),
    ("Non-Complex Case Count", int((data["complex_case_status"] == 0).sum()) if "complex_case_status" in data.columns else "N/A"),
    ("N Unique Surgeons", int(data[surgeon_col].nunique()) if surgeon_col in data.columns else "N/A"),
    ("Project", "Prompt-Technique Evaluation for Feature-Level Human vs LLM Validation (Base Model)"),
    ("Repository", "https://github.com/Robertjam954/AI-Jupyter-Notebook.git"),
])

df_metadata = pd.DataFrame(list(metadata.items()), columns=["Attribute", "Value"])
print("Dataset Metadata:")
df_metadata

Dataset Metadata:


,Attribute,Value
0,Dataset Name,Merged LLM Summary Validation Datasheet (Deide...
1,Source File,merged_llm_summary_validation_datasheet.xlsx
2,Date Generated,2026-03-02 20:25:55
3,N Observations,200
4,N Variables,49
5,N Elements (Features),14
6,N Radiology Elements,33
7,N Pathology Elements,12
8,Annotator Roles,"Source (Ground Truth), Human Annotator, AI (LL..."
9,Source Coding,"0 = Feature absent in source document, 1 = Fea..."


In [7]:
# Per-element metadata summary
element_meta_rows = []
for elem, cols in ELEMENTS.items():
    s_col = cols["source"]
    h_col = cols["human"]
    a_col = cols["ai"]
    domain = DOMAIN_LOOKUP.get(s_col, "")
    n_present = int((data[s_col] == 1).sum()) if s_col in data.columns else 0
    n_absent = int((data[s_col] == 0).sum()) if s_col in data.columns else 0
    h_miss = int(data[h_col].isna().sum()) if h_col in data.columns else data.shape[0]
    a_miss = int(data[a_col].isna().sum()) if a_col in data.columns else data.shape[0]
    element_meta_rows.append({
        "Element": elem,
        "Domain": domain,
        "Source Column": s_col,
        "Human Column": h_col,
        "AI Column": a_col,
        "N Source Present (1)": n_present,
        "N Source Absent (0)": n_absent,
        "Prevalence (%)": round(n_present / data.shape[0] * 100, 1),
        "Human Missing (N)": h_miss,
        "AI Missing (N)": a_miss,
    })

df_element_meta = pd.DataFrame(element_meta_rows)
print("\nPer-Element Metadata:")
df_element_meta


Per-Element Metadata:


,Element,Domain,Source Column,Human Column,AI Column,N Source Present (1),N Source Absent (0),Prevalence (%),Human Missing (N),AI Missing (N)
0,Lesion Size,Radiology,lesion_size_status_source,lesion_size_status_human,lesion_size_status_ai,196,4,98.0,2,2
1,Lesion Laterality,Radiology,laterality_status_source,laterality_status_human,laterality_status_ai,200,0,100.0,0,0
2,Lesion Location,Radiology,lesion_location_status_source,lesion_location_status_human,lesion_location_status_ai,199,1,99.5,1,1
3,Calcifications / Asymmetry,Radiology,calcifications_asymmetry_status_source,calcifications_asymmetry_status_human,calcifications_asymmetry_status_ai,127,73,63.5,59,59
4,Additional Enhancement (MRI),Radiology,additional_enhancement_mri_status_source,additional_enhancement_mri_status_human,additional_enhancement_mri_status_ai,57,143,28.5,108,108
5,Extent,Radiology,extent_status_source,extent_status_human,extent_status_ai,88,112,44.0,76,76
6,Accurate Clip Placement,Radiology,accurate_clip_placement_status_source,accurate_clip_placement_status_human,accurate_clip_placement_status_ai,180,20,90.0,16,18
7,Workup Recommendation,Radiology,workup_recommendation_status_source,workup_recommendation_status_human,workup_recommendation_status_ai,161,39,80.5,38,38
8,Lymph Node,Radiology,Lymph node_status_source,Lymph node_status_human,Lymph node_status_ai,163,37,81.5,22,22
9,Chronology Preserved,Radiology,chronology_preserved_status_source,chronology_preserved_status_human,chronology_preserved_status_ai,199,1,99.5,1,1


---
## 5. Export to Excel

In [8]:
# --- Data Dictionary Excel ---
dict_path = OUTPUT_DIR / "data_dictionary.xlsx"
with pd.ExcelWriter(dict_path, engine="openpyxl") as writer:
    df_dict.to_excel(writer, sheet_name="Data Dictionary", index=False)
    df_element_meta.to_excel(writer, sheet_name="Element Summary", index=False)
    df_metadata.to_excel(writer, sheet_name="Dataset Metadata", index=False)
print(f"Data Dictionary saved: {dict_path}")

# --- Variable Names Excel ---
var_path = OUTPUT_DIR / "variable_names.xlsx"
with pd.ExcelWriter(var_path, engine="openpyxl") as writer:
    df_vars.to_excel(writer, sheet_name="Variable Names", index=False)
print(f"Variable Names saved: {var_path}")

Data Dictionary saved: C:\Users\jamesr4\OneDrive - Memorial Sloan Kettering Cancer Center\Documents\GitHub\llm_summarization_br_ca\reports\data_dictionary.xlsx


Variable Names saved: C:\Users\jamesr4\OneDrive - Memorial Sloan Kettering Cancer Center\Documents\GitHub\llm_summarization_br_ca\reports\variable_names.xlsx


In [9]:
# Format the Excel files with column widths and header styling
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

def style_excel(filepath, sheet_configs=None):
    """Apply consistent formatting to an Excel workbook."""
    wb = load_workbook(filepath)
    header_font = Font(bold=True, color="FFFFFF", size=11)
    header_fill = PatternFill(start_color="2C3E50", end_color="2C3E50", fill_type="solid")
    header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
    cell_align = Alignment(vertical="top", wrap_text=True)
    thin_border = Border(
        left=Side(style="thin"),
        right=Side(style="thin"),
        top=Side(style="thin"),
        bottom=Side(style="thin"),
    )

    for ws in wb.worksheets:
        # Auto-size columns (approximate)
        for col in ws.columns:
            max_length = 0
            col_letter = col[0].column_letter
            for cell in col:
                try:
                    if cell.value:
                        max_length = max(max_length, len(str(cell.value)))
                except:
                    pass
            adjusted_width = min(max_length + 4, 50)
            ws.column_dimensions[col_letter].width = adjusted_width

        # Style header row
        for cell in ws[1]:
            cell.font = header_font
            cell.fill = header_fill
            cell.alignment = header_align
            cell.border = thin_border

        # Style data rows
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = cell_align
                cell.border = thin_border

        # Freeze header row
        ws.freeze_panes = "A2"

    wb.save(filepath)
    print(f"  Styled: {filepath}")

style_excel(dict_path)
style_excel(var_path)
print("\nDone. All Excel files styled and saved.")

  Styled: C:\Users\jamesr4\OneDrive - Memorial Sloan Kettering Cancer Center\Documents\GitHub\llm_summarization_br_ca\reports\data_dictionary.xlsx
  Styled: C:\Users\jamesr4\OneDrive - Memorial Sloan Kettering Cancer Center\Documents\GitHub\llm_summarization_br_ca\reports\variable_names.xlsx

Done. All Excel files styled and saved.


---
## 6. Quick Validation

In [10]:
# Verify outputs
print("=" * 60)
print("OUTPUT FILES")
print("=" * 60)
for f in [dict_path, var_path]:
    if f.exists():
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:40s} {size_kb:7.1f} KB")
    else:
        print(f"  {f.name:40s} MISSING")

print(f"\nData Dictionary:")
print(f"  Sheets: Data Dictionary ({len(df_dict)} rows), Element Summary ({len(df_element_meta)} rows), Dataset Metadata ({len(df_metadata)} rows)")
print(f"  Headers: {list(df_dict.columns)}")
print(f"\nVariable Names:")
print(f"  Sheets: Variable Names ({len(df_vars)} rows)")
print(f"  Headers: {list(df_vars.columns)}")

OUTPUT FILES
  data_dictionary.xlsx                        13.0 KB
  variable_names.xlsx                          7.9 KB

Data Dictionary:
  Sheets: Data Dictionary (49 rows), Element Summary (14 rows), Dataset Metadata (22 rows)
  Headers: ['Variable Name', 'Label', 'Description', 'Data Type', 'Valid Values', 'Source Column', 'Domain', 'Role', 'Missing Code', 'Notes']

Variable Names:
  Sheets: Variable Names (49 rows)
  Headers: ['Position', 'Variable Name', 'Label', 'Data Type', 'Domain', 'Role', 'N Unique', 'N Missing', 'Pct Missing']
